In [ ]:
# check
import geopandas as gpd
import pandas as pd
import contextily as cx
import yaml
import matplotlib.pyplot as plt
import rasterio
from rasterio.plot import show
from rasterio import plot as rasterplot
from rasterio.plot import show_hist
from rasterio.mask import mask
import os
import numpy as np
from cafo_iowa.data.preprocess_shp import add_buffer_to_tiles
from shapely.wkt import loads
from mpl_toolkits.axes_grid1 import make_axes_locatable

In [ ]:
# load config
with open("cafo_iowa/config_old.yaml", "r") as f:
    config = yaml.safe_load(f)

In [ ]:
# load data
tiles = gpd.read_feather("data/NAIP21QQ_shp/processed/02_cropped_tiles.feather")
quartered_tiles = gpd.read_feather(
    "data/NAIP21QQ_shp/processed/03_quartered_tiles.feather"
)
quartered_buffer_tiles = gpd.read_feather(
    "data/NAIP21QQ_shp/processed/04_quartered_buffer_tiles.feather"
)
permits = pd.read_csv("data/permits/processed/permits.csv")

In [ ]:
# save permits as geodataframe
permits = gpd.GeoDataFrame(
    permits, geometry=gpd.points_from_xy(permits.longitude, permits.latitude)
)
permits.crs = {"init": "epsg:4326"}
permits = permits.to_crs(tiles.crs)

In [ ]:
# subset permits to only those columns that start with swine and that have more than 0 animals
swine_cols = [col for col in permits.columns if col.startswith("swine")]
swine_permits = permits[permits[swine_cols].sum(axis=1) > 0]
swine_permits.shape

In [ ]:
# join permits to qt_tiles
quartered_permit_tiles = gpd.sjoin(
    quartered_tiles, swine_permits, how="left", op="intersects"
)

print(quartered_permit_tiles.columns)
# count permits in each tile
quartered_permit_tiles = (
    quartered_permit_tiles.dissolve(
        by="qt_tile_id", aggfunc={"fid": "count", "totalanima": "sum"}
    )
    .rename(columns={"fid": "permit_count", "totalanima": "animal_count"})
    .reset_index()
)

In [ ]:
# merge permits to quartered buffer tiles
permits_quartered_locs = gpd.sjoin(
    swine_permits, quartered_tiles, how="left", op="intersects"
)
permits_quartered_locs

In [ ]:
print(
    "Number of tiles:",
    len(permits),
    "\nNumber of tiles:",
    len(tiles),
    "\nNumber of qt tiles:",
    len(quartered_tiles),
    "\nNumber of qt permit tiles:",
    len(quartered_permit_tiles),
    "\nNumber of permits:",
    len(permits),
    "\nNumber of swine permits:",
    len(swine_permits),
    "\nUnique qt tiles with 1+ permit:",
    len(quartered_permit_tiles[quartered_permit_tiles["permit_count"] > 0]),
    "\nAverage permits per tile (nonzero):",
    round(
        np.mean(
            quartered_permit_tiles[quartered_permit_tiles["permit_count"] > 0][
                "permit_count"
            ]
        ),
        2,
    ),
    "\nAverage animals per tile (nonzero):",
    round(
        np.mean(
            quartered_permit_tiles[quartered_permit_tiles["permit_count"] > 0][
                "animal_count"
            ]
        ),
        2,
    ),
)

In [ ]:
# plot tiles
p1 = tiles.plot(edgecolor="red", linewidth=0.1, facecolor="none")
p1.axis("off")
cx.add_basemap(p1, crs=tiles.crs, source=cx.providers.CartoDB.Positron)
p1.get_figure().savefig(
    config.get("plots").get("output_path") + "tiles.png",
    dpi=config.get("plots").get("dpi"),
    bbox_inches=config.get("plots").get("bbox_inches"),
    pad_inches=0,
)

In [ ]:
# Directory containing the GeoTIFF files
fp = "data/NAIP21QQ/01_raw_tiles/tifs/"
files = os.listdir(fp)[0:2]

# select tiles with corresponding tile_ids
shp_tiles = tiles[tiles["tile_id"].isin([file.replace(".tif", "") for file in files])]
shp_tiles_quartered = quartered_tiles[
    quartered_tiles["tile_id"].isin([file.replace(".tif", "") for file in files])
]
shp_tiles_quartered_buffer = quartered_buffer_tiles[
    quartered_tiles["tile_id"].isin([file.replace(".tif", "") for file in files])
]

# subset quartered tiles where the index ends with "_BR"
shp_tiles_quartered = shp_tiles_quartered[
    shp_tiles_quartered.qt_tile_id.str.endswith("_BR")
]
shp_tiles_quartered_buffer = shp_tiles_quartered_buffer[
    shp_tiles_quartered_buffer.qt_tile_id.str.endswith("_BR")
]

# Number of rows and columns for subplots
num_rows = 1
num_cols = len(files)

# # Create a figure and axes for subplots
fig, axes = plt.subplots(num_rows, num_cols, figsize=(7, 5))

# Plot geotiffs with shapefile overlay
for i, file in enumerate(files):
    if file.endswith(".tif"):
        raster_path = os.path.join(fp, file)
        raster = rasterio.open(raster_path)
        shp = shp_tiles[shp_tiles["tile_id"] == file.replace(".tif", "")]
        quartered_shp = shp_tiles_quartered[
            shp_tiles_quartered["tile_id"] == file.replace(".tif", "")
        ]
        quartered_buffer_shp = shp_tiles_quartered_buffer[
            shp_tiles_quartered["tile_id"] == file.replace(".tif", "")
        ]
        ax = axes[i] if num_cols > 1 else axes
        show(raster, ax=ax)
        shp.plot(ax=ax, facecolor="none", edgecolor="red", linewidth=1)
        quartered_shp.plot(ax=ax, facecolor="none", edgecolor="blue", linewidth=0.5)
        quartered_buffer_shp.geometry.plot(
            ax=ax, facecolor="none", edgecolor="pink", linewidth=0.5
        )
        ax.axis("off")

plt.tight_layout()
plt.show()
fig.savefig(
    config.get("plots").get("output_path") + "geotiffs4.png",
    dpi=config.get("plots").get("dpi"),
    bbox_inches=config.get("plots").get("bbox_inches"),
    pad_inches=0,
)

In [ ]:
# plot quarter tiles
p2 = quartered_tiles.plot(edgecolor="blue", linewidth=0.05, facecolor="none")
p2.axis("off")
tiles.plot(ax=p2, edgecolor="red", linewidth=0.1, facecolor="none")
cx.add_basemap(p2, crs=quartered_tiles.crs, source=cx.providers.CartoDB.Positron)
p2.get_figure().savefig(
    config.get("plots").get("output_path") + "qt_tiles.png",
    dpi=config.get("plots").get("dpi"),
    bbox_inches=config.get("plots").get("bbox_inches"),
    pad_inches=0,
)

In [ ]:
# plot permit count and animal count per tile
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].scatter(
    quartered_permit_tiles["permit_count"],
    quartered_permit_tiles["animal_count"],
    alpha=0.2,
    facecolor="maroon",
)
ax[0].set_title("Permits and animals per tile")
ax[0].set_xlabel("Permit Count")
ax[0].set_ylabel("Animal Count")
ax[0].set_yscale("log")
ax[1].hist(quartered_permit_tiles["permit_count"], bins=20, facecolor="maroon")
ax[1].set_title("Permits per tile")
plt.show()
fig.savefig(
    config.get("plots").get("output_path") + "permits+animals-per-qttile.png",
    dpi=config.get("plots").get("dpi"),
    bbox_inches=config.get("plots").get("bbox_inches"),
    pad_inches=0,
)

In [ ]:
# animals per permit
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].hist(swine_permits["totalanima"], facecolor="maroon", bins=100)
# log scale
ax[0].set_yscale("log")
# title and labels
ax[0].set_title("Animals per permit")
ax[0].set_xlabel("Animal Count")
ax[0].set_ylabel("Frequency")
# subset second plot to only show permits with less than 15k animals
ax[1].hist(
    swine_permits[swine_permits["totalanima"] < 15000]["totalanima"],
    facecolor="blue",
    bins=100,
)
ax[1].set_title("Animals per permit (less than 15k)")
ax[1].set_xlabel("Animal Count")
ax[1].set_ylabel("Frequency")

fig.savefig(
    config.get("plots").get("output_path") + "animals-per-permit.png",
    dpi=config.get("plots").get("dpi"),
    bbox_inches=config.get("plots").get("bbox_inches"),
    pad_inches=0,
)

In [ ]:
# Create subplots with 1 row and 2 columns
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))

# First map - plot quartered tiles with permit count
ax1.set_title("Permit Count")
divider1 = make_axes_locatable(ax1)
cax1 = divider1.append_axes("bottom", size="5%", pad=0.1)
quartered_permit_tiles[quartered_permit_tiles["permit_count"] > 0].plot(
    colormap="viridis",
    column="permit_count",
    ax=ax1,
    legend=True,
    linewidth=0,
    edgecolor="none",
    cax=cax1,
    legend_kwds={"label": "Number of permits per tile", "orientation": "horizontal"},
)

# Add basemap to the first map
cx.add_basemap(
    ax1, crs=quartered_permit_tiles.crs, source=cx.providers.CartoDB.Positron
)

ax1.axis("off")

# # Second map - differentiate between permit count classes
ax2.set_title("Permit Count (Discrete)")
quartered_permit_tiles["permit_count_cat"] = pd.cut(
    quartered_permit_tiles["permit_count"],
    bins=[0, 1, 2, 3, 4, 100],
    labels=["0", "1", "2", "3", "4+"],
    right=False,
)
ax2.plot()  # Placeholder for your second map
divider2 = make_axes_locatable(ax2)
quartered_permit_tiles[quartered_permit_tiles["permit_count"] > 0].plot(
    colormap="viridis",
    column="permit_count_cat",
    ax=ax2,
    linewidth=0,
    edgecolor="none",
    legend=True,
)

# Add basemap to the first map
cx.add_basemap(
    ax2, crs=quartered_permit_tiles.crs, source=cx.providers.CartoDB.Positron
)

ax2.axis("off")

# Save the figure
plt.show()
fig.savefig(
    config.get("plots").get("output_path") + "map-permits-per-tile.png",
    dpi=config.get("plots").get("dpi"),
    bbox_inches=config.get("plots").get("bbox_inches"),
    pad_inches=0,
)

In [ ]:
# get the quartered buffer tile with the highest number of permits
max_permit_tile = quartered_permit_tiles[
    quartered_permit_tiles["permit_count"]
    == quartered_permit_tiles["permit_count"].max()
]

# select corresponding permit locations
permit_locs = permits_quartered_locs[
    permits_quartered_locs["qt_tile_id"] == max_permit_tile["qt_tile_id"].values[0]
]

# open the images with rasterio
max_tile = max_permit_tile["qt_tile_id"].values[0]
fp = "data/NAIP21QQ/04_quartered_buffer_tiles/tifs/"

img = rasterio.open(fp + max_tile + ".tif")
# show the image
fig, ax = plt.subplots(1, 1, figsize=(10, 10))
show(img, ax=ax)
# remove axis
ax.axis("off")
# add the quartered buffer tile on top
max_permit_tile.plot(ax=ax, facecolor="none", edgecolor="blue", linewidth=1)
# add the permit locations on top
permit_locs.plot(ax=ax, color="red", markersize=5)
# save
plt.show()
fig.savefig(
    config.get("plots").get("output_path") + "tile_with_most_permits.png",
    dpi=config.get("plots").get("dpi"),
    bbox_inches=config.get("plots").get("bbox_inches"),
    pad_inches=0,
)

In [ ]:
# get the quartered buffer tile with the highest number of permits
max_animal_tile = quartered_permit_tiles[
    quartered_permit_tiles["animal_count"]
    == quartered_permit_tiles["animal_count"].max()
]
print(max_animal_tile["animal_count"].values[0])
# select corresponding permit locations
permit_locs = permits_quartered_locs[
    permits_quartered_locs["qt_tile_id"] == max_animal_tile["qt_tile_id"].values[0]
]

# open the images with rasterio
max_tile = max_animal_tile["qt_tile_id"].values[0]
fp = "data/NAIP21QQ/04_quartered_buffer_tiles/tifs/"

img = rasterio.open(fp + max_tile + ".tif")
# show the image
fig, ax = plt.subplots(1, 1, figsize=(10, 10))
show(img, ax=ax)
# remove axis
ax.axis("off")
# add the quartered buffer tile on top
max_animal_tile.plot(ax=ax, facecolor="none", edgecolor="blue", linewidth=1)
# add the permit locations on top
permit_locs.plot(ax=ax, color="red", markersize=5)
# save
plt.show()
fig.savefig(
    config.get("plots").get("output_path") + "tile_with_most_animals.png",
    dpi=config.get("plots").get("dpi"),
    bbox_inches=config.get("plots").get("bbox_inches"),
    pad_inches=0,
)